# 55 - Refreshed silver labels, up to 1,000 per query, no new cost

Notebooks 39/41 already distill the gold standard into a classifier and apply it to every existing candidate for free. This refreshes that with the richest gold standard available so far (2,269 labels: the 1,429-candidate 101-query expansion from notebook 40, plus the 840 extra labels the 14 headline queries picked up from notebook 42's active-learning rounds), and caps the output at up to 1,000 candidates per query rather than keeping the entire pool.

No new retrieval, no new encoding, no new API calls: every query already has at least 1,022 candidates sitting in `scored_candidates.csv` (median 1,661), so this only needs to re-run classifier inference, not wait on notebooks 44-54's scaled encoding. Still silver, not gold, capped by the classifier's demonstrated accuracy (83% on the headline queries at their current depth, ~80.7% at the prior full-101-query check in notebook 41), so this is deep per-query coverage for downstream use, not a replacement for the gold labels themselves.

In [1]:
import json
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

OUTPUT_DIR = Path("result/55_silver_labels_refreshed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RELEVANT_THRESHOLD = 2  # same strict "highly relevant" definition used throughout
TOP_K_PER_QUERY = 1000

feature_cols = ["score_minilm", "score_linq", "score_gte", "score_bm25",
                "invrank_minilm", "invrank_linq", "invrank_gte", "invrank_bm25",
                "reranker_score", "n_channels"]

base_gold = pd.read_json("result/40_active_learning_labeling_queue/expanded_gold_labels.json")
round_gold = pd.read_json("result/42_headline_query_deepening/round_gold_labels.json")
features = pd.read_csv("result/30_learned_fusion_ranker/scored_candidates.csv")

gold = pd.concat([
    base_gold[["query_id", "domain", "gold_label"]],
    round_gold[["query_id", "domain", "gold_label"]],
], ignore_index=True)
n_before = len(gold)
gold = gold.drop_duplicates(subset=["query_id", "domain"])
print(f"Combined gold standard: {len(gold)} candidates ({n_before - len(gold)} duplicates dropped)")
print(f"Queries covered: {gold['query_id'].nunique()}/101")

labeled = gold.merge(features, on=["query_id", "domain"], how="inner")
labeled["relevant"] = (labeled["gold_label"] >= RELEVANT_THRESHOLD).astype(int)
print(f"Matched to features: {len(labeled)}/{len(gold)}")
print(f"Relevant (gold_label >= {RELEVANT_THRESHOLD}): {labeled['relevant'].sum()}/{len(labeled)}")

Combined gold standard: 2269 candidates (0 duplicates dropped)
Queries covered: 101/101
Matched to features: 2267/2269
Relevant (gold_label >= 2): 1499/2267


In [2]:
# Deployment model: fit on ALL available gold labels, no held-out split (same reasoning as
# notebooks 39/41 -- this model is meant to be applied at scale, not evaluated in isolation).
X_train = labeled[feature_cols].values
y_train = labeled["relevant"].values

final_gbdt = HistGradientBoostingClassifier(max_iter=150, max_depth=4, class_weight="balanced", random_state=0)
final_gbdt.fit(X_train, y_train)

final_logreg = LogisticRegression(max_iter=2000, class_weight="balanced")
final_logreg.fit(X_train, y_train)

print(f"Final model trained on all {len(labeled)} gold-labeled candidates spanning {labeled['query_id'].nunique()} queries.")
print("Feature weights (logistic regression, for reference):")
for name, coef in sorted(zip(feature_cols, final_logreg.coef_[0]), key=lambda x: -abs(x[1])):
    print(f"    {name:<16} {coef:+.3f}")

Final model trained on all 2267 gold-labeled candidates spanning 101 queries.
Feature weights (logistic regression, for reference):
    invrank_bm25     -2.220
    invrank_minilm   -1.403
    score_linq       +1.195
    invrank_linq     +0.343
    invrank_gte      -0.184
    reranker_score   +0.112
    score_minilm     -0.087
    n_channels       +0.085
    score_gte        +0.045
    score_bm25       -0.015


In [3]:
# Score every existing candidate -- zero additional retrieval/encoding cost, reuses scored_candidates.csv.
X_all = features[feature_cols].values
features["silver_prob_relevant"] = final_gbdt.predict_proba(X_all)[:, 1]

gold_keys = set(zip(gold["query_id"], gold["domain"]))
features["tier"] = ["gold" if (q, d) in gold_keys else "silver" for q, d in zip(features["query_id"], features["domain"])]

# Cap at TOP_K_PER_QUERY per query, but never drop an already-verified gold candidate to make
# room for a merely-predicted silver one -- gold rows always win the tie-break within a query.
features["_sort_key"] = features["tier"].map({"gold": 1, "silver": 0}).astype(float) + features["silver_prob_relevant"]
capped = (
    features.sort_values("_sort_key", ascending=False)
    .groupby("query_id", group_keys=False)
    .head(TOP_K_PER_QUERY)
    .drop(columns="_sort_key")
    .sort_values(["query_id", "silver_prob_relevant"], ascending=[True, False])
    .reset_index(drop=True)
)

out_path = OUTPUT_DIR / "silver_labels_top1000.json"
capped.to_json(out_path, orient="records", indent=2)

per_query = capped.groupby("query_id").size()
print(f"Saved -> {out_path}")
print(f"Total rows: {len(capped)}")
print(f"Candidates per query: min={per_query.min()}, median={per_query.median():.0f}, max={per_query.max()}")
print(f"Tier breakdown:\n{capped['tier'].value_counts()}")

Saved -> result/55_silver_labels_refreshed/silver_labels_top1000.json
Total rows: 101000
Candidates per query: min=1000, median=1000, max=1000
Tier breakdown:
tier
silver    98733
gold       2267
Name: count, dtype: int64


## How does production's own ranking compare to the independent gold/silver set?

`dataset/production_results.xlsx` is production's own top-1000-per-query ranking, exactly the same shape as `silver_labels_top1000.json` just built (101,000 rows, 1,000 per query, same `query_id`/`domain` keys), which makes a direct comparison possible with no new judging or encoding. Four checks: (1) raw overlap between the two top-1000 lists, (2) how often production's own picks are actually confirmed relevant by the independent gold labels, (3) how many gold-verified *highly relevant* companies production's ranking misses entirely, and (4) whether production's rank order correlates with the independently-trained classifier's predicted probability where the two overlap.

In [4]:
production = pd.read_excel("dataset/production_results.xlsx")[["query_id", "domain", "rank"]].rename(columns={"rank": "production_rank"})
ours = pd.read_json(OUTPUT_DIR / "silver_labels_top1000.json")[["query_id", "domain", "tier", "silver_prob_relevant"]]

# Re-attach gold_label to the gold-tier rows -- capped/ours only carried the tier flag, not the label itself.
gold_labels_lookup = gold.set_index(["query_id", "domain"])["gold_label"]
ours["gold_label"] = ours.set_index(["query_id", "domain"]).index.map(gold_labels_lookup).values

print(f"Production ranking : {len(production):,} rows, {production['query_id'].nunique()} queries")
print(f"Our top-1000 set    : {len(ours):,} rows, {ours['query_id'].nunique()} queries")

Production ranking : 101,000 rows, 101 queries
Our top-1000 set    : 101,000 rows, 101 queries


In [5]:
merged = production.merge(ours, on=["query_id", "domain"], how="outer", indicator=True)

overlap_by_query = merged.groupby("query_id")["_merge"].value_counts().unstack(fill_value=0)
overlap_by_query["overlap_pct"] = overlap_by_query["both"] / 1000 * 100

print("Check 1: overlap between production's top-1000 and our top-1000, per query")
print(f"Mean overlap    : {overlap_by_query['overlap_pct'].mean():.1f}%")
print(f"Median overlap  : {overlap_by_query['overlap_pct'].median():.1f}%")
print(f"Min / max       : {overlap_by_query['overlap_pct'].min():.1f}% / {overlap_by_query['overlap_pct'].max():.1f}%")
print()
print("5 queries with the LOWEST overlap (most divergent from production):")
print(overlap_by_query.sort_values("overlap_pct").head(5)[["overlap_pct"]])

Check 1: overlap between production's top-1000 and our top-1000, per query
Mean overlap    : 77.2%
Median overlap  : 81.0%
Min / max       : 34.9% / 99.6%

5 queries with the LOWEST overlap (most divergent from production):
_merge    overlap_pct
query_id             
1                34.9
3                36.4
73               43.1
19               43.9
25               44.2


In [6]:
production_with_gold = merged[merged["production_rank"].notna() & merged["gold_label"].notna()]

print("Check 2: of production's top-1000 picks that happen to have an independent gold label,")
print("what does that label actually say?")
print(f"Production picks with a gold label available: {len(production_with_gold)}")
print()
print(production_with_gold["gold_label"].value_counts(normalize=True).sort_index().rename(
    {0: "0 = not relevant", 1: "1 = partially relevant", 2: "2 = highly relevant"}
).mul(100).round(1))

Check 2: of production's top-1000 picks that happen to have an independent gold label,
what does that label actually say?
Production picks with a gold label available: 1434

gold_label
0 = not relevant           1.8
1 = partially relevant    22.2
2 = highly relevant       76.0
Name: proportion, dtype: float64


In [7]:
gold_relevant = merged[merged["gold_label"] == 2]
missed_by_production = gold_relevant[gold_relevant["production_rank"].isna()]

print("Check 3: of gold-verified HIGHLY RELEVANT companies, how many does production's")
print("own top-1000 ranking miss entirely?")
print(f"Gold-verified highly relevant candidates : {len(gold_relevant)}")
print(f"Missing from production's top-1000       : {len(missed_by_production)} ({100*len(missed_by_production)/len(gold_relevant):.1f}%)")
print()
print("By query (queries with the most missed highly-relevant companies):")
miss_by_query = missed_by_production.groupby("query_id").size().sort_values(ascending=False)
print(miss_by_query.head(10))

Check 3: of gold-verified HIGHLY RELEVANT companies, how many does production's
own top-1000 ranking miss entirely?
Gold-verified highly relevant candidates : 1499
Missing from production's top-1000       : 409 (27.3%)

By query (queries with the most missed highly-relevant companies):
query_id
92    45
91    40
1     27
15    26
66    21
72    14
14    13
4     12
56    10
79     9
dtype: int64


In [8]:
both = merged[merged["production_rank"].notna() & merged["silver_prob_relevant"].notna()].copy()

# Better production rank = lower number; better silver score = higher number, so agreement shows
# up as a NEGATIVE Spearman correlation between production_rank and silver_prob_relevant.
per_query_corr = both.groupby("query_id").apply(
    lambda g: g["production_rank"].corr(g["silver_prob_relevant"], method="spearman") if len(g) > 5 else None
)
per_query_corr = per_query_corr.dropna()

print("Check 4: does production's rank order agree with the independently-trained classifier's")
print("predicted probability, for candidates both systems consider?")
print(f"Candidates compared        : {len(both):,}")
print(f"Queries with enough overlap: {len(per_query_corr)}/101")
print(f"Mean per-query Spearman correlation (production_rank vs silver_prob_relevant): {per_query_corr.mean():+.3f}")
print("(negative = agreement, i.e. better production rank -> higher predicted relevance; near zero or positive = disagreement)")
print()
print("5 queries where the two rankings disagree the MOST (least negative / most positive correlation):")
print(per_query_corr.sort_values(ascending=False).head(5))

Check 4: does production's rank order agree with the independently-trained classifier's
predicted probability, for candidates both systems consider?
Candidates compared        : 77,993
Queries with enough overlap: 101/101
Mean per-query Spearman correlation (production_rank vs silver_prob_relevant): -0.056
(negative = agreement, i.e. better production rank -> higher predicted relevance; near zero or positive = disagreement)

5 queries where the two rankings disagree the MOST (least negative / most positive correlation):
query_id
29    0.289685
75    0.240761
13    0.158582
3     0.120114
14    0.100616
dtype: float64


/scratch/ipykernel_2372182/2036169648.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_query_corr = both.groupby("query_id").apply(
